# HVAC Takeoff — v9 Training on Colab (124 projects, 25K tiles)

**Steps:**
1. Upload `yolo_dataset.zip` (1.27 GB) to your **Google Drive root**
2. Set runtime to **T4 GPU**
3. Run all cells

The notebook mounts Drive, copies the zip, extracts, and trains.

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics -q

In [ ]:
# Step 2: Mount Google Drive and extract dataset
from google.colab import drive
import zipfile, os, yaml, shutil

drive.mount('/content/drive')

# Copy zip from Drive to local (faster I/O)
src = '/content/drive/MyDrive/yolo_dataset.zip'
if not os.path.exists(src):
    print(f'ERROR: {src} not found!')
    print('Upload yolo_dataset.zip to the ROOT of your Google Drive first.')
else:
    print(f'Found {src} ({os.path.getsize(src)/1024/1024:.0f} MB)')
    print('Copying to local disk...')
    shutil.copy(src, '/content/yolo_dataset.zip')
    
    print('Extracting...')
    with zipfile.ZipFile('/content/yolo_dataset.zip', 'r') as z:
        z.extractall('/content/')
    
    # Fix paths in dataset.yaml
    config_path = '/content/yolo_dataset/dataset.yaml'
    with open(config_path) as f:
        config = yaml.safe_load(f)
    config['path'] = '/content/yolo_dataset'
    with open(config_path, 'w') as f:
        yaml.dump(config, f)
    
    train_imgs = len(os.listdir('/content/yolo_dataset/images/train'))
    val_imgs = len(os.listdir('/content/yolo_dataset/images/val'))
    print(f'\nDataset: {train_imgs} train, {val_imgs} val images')
    print(f'Classes: {len(config["names"])}')
    for i, n in config['names'].items():
        print(f'  {i}: {n}')

In [ ]:
# Step 3: Train YOLOv8s
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=80,         # Fewer epochs since dataset is 10x bigger
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    workers=2,
    project='/content/runs',
    name='hvac_v9',
    exist_ok=True,
    flipud=0.0,
    mosaic=0.5,
    scale=0.3,
)

print('\nTraining complete!')

In [ ]:
# Step 4: Check results
import pandas as pd

df = pd.read_csv('/content/runs/hvac_detect/results.csv')
df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best['epoch'])}")
print(f"Precision: {best['metrics/precision(B)']:.1%}")
print(f"Recall:    {best['metrics/recall(B)']:.1%}")
print(f"mAP50:     {best['metrics/mAP50(B)']:.1%}")
print(f"mAP50-95:  {best['metrics/mAP50-95(B)']:.1%}")

In [ ]:
# Step 5: Download trained model + save to Drive
from google.colab import files
import shutil

best = '/content/runs/hvac_v9/weights/best.pt'
# Save to Drive for backup
shutil.copy(best, '/content/drive/MyDrive/hvac_yolov8s_v9.pt')
print('Saved to Google Drive: hvac_yolov8s_v9.pt')

# Also download to browser
shutil.copy(best, '/content/hvac_yolov8s_v9.pt')
files.download('/content/hvac_yolov8s_v9.pt')
print('Download complete — put it in hvac-takeoff-tool/models/')